# HMApp — inspect a completed Raiders experiment

Change **RUN_DIRECTORY** below and run all cells. This notebook only reads completed
artifacts. Extract the downloaded ZIP into `artifacts/` first; **RUN_DIRECTORY** must point to
the extracted folder containing `run_info.json`, not the ZIP file.
It does not import BERT, generate embeddings, download resources or require a GPU.
The original ranking is preserved; the additional views are inspection filters, not improved
performance claims. Charts use plain HTML, like the reference-preparation notebook, so no
new plotting dependency or internet connection is needed.

## 1. Run overview

In [50]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src/utils/results.py").is_file():
    raise RuntimeError("Open this notebook from the repository root or src directory.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from utils.results import load_run, inspection_view, phrase_pattern, highlight_word_spans, bar_chart_html

RUN_DIRECTORY = PROJECT_ROOT / "artifacts/raiders_islamophobia_20261030T103802_739674Z"

run_info, tables, candidate_stage_counts = load_run(RUN_DIRECTORY)
posts = tables["discovery_posts"]
rankings = tables["ranked_candidates"]
occurrences = tables["candidate_occurrences"]
post_scores = tables["candidate_post_scores"]
seed_words = run_info["seed_words"]
anchor_variants = list(dict.fromkeys(
    variant for variants in run_info["anchor_terms"].values() for variant in variants
))

display(pd.Series({
    "run": str(RUN_DIRECTORY), "started_utc": run_info["started_at_utc"],
    "source": run_info["retrieval"]["source_path"],
    "reference_bank": run_info["reference_bank"]["path"],
    "reference_bank_id": run_info["reference_bank"]["identifier_sha256"],
    "checkpoint": run_info["embedding_config"]["checkpoint"],
    "model_revision": run_info["embedding_config"]["resolved_revision"],
    "context_tokens_each_side": run_info["embedding_config"]["context_tokens_each_side"],
    "batch_size": run_info["candidate_batch_size"], "device": run_info["device"],
    "calculation_seconds": run_info["calculation_seconds"],
}))
print("Retrieval seeds:", seed_words)
print("Seed provenance:", run_info["seed_provenance"])
print("Separation:", run_info["retrieval"]["separation"])
display(pd.Series(run_info["stage_seconds"], name="seconds").to_frame())
reference_coverage = tables["reference_coverage"].set_index("anchor")
display(reference_coverage)
display(HTML(bar_chart_html(reference_coverage["reference_vectors"].sort_values(ascending=False),
                            "Fixed reference vectors per anchor")))

run                         /home/armand/MS_IA/IA717/project/Emerging_Code...
started_utc                                  2026-10-30T11:01:21.860599+00:00
source                      /content/hmapp_cad2ca2867f74c12ada8d9ba760c6d5...
reference_bank              /content/hmapp_cad2ca2867f74c12ada8d9ba760c6d5...
reference_bank_id           e898c2c34066c932dbfc222a8d7631994d2d96314ea6ca...
checkpoint                                      google-bert/bert-base-uncased
model_revision                       86b5e0934494bd15c9632b12f734a8a67f723594
context_tokens_each_side                                                   64
batch_size                                                                 16
device                                                                   cuda
calculation_seconds                                                684.205004
dtype: object

Retrieval seeds: ['islam', 'muslim', 'muslims', 'islamic', 'mosque', 'mosques', 'quran', 'koran', 'islamophobia']
Seed provenance: Experimental seeds from vocabulary.py for islamophobia; not Wu's published seed selection
Separation: Exclude every thread represented in the fixed bank; then exclude normalized reference text


,seconds
reference_loading,0.034497
retrieval,30.679196
candidate_preparation,31.545552
model_loading,5.424518
candidate_embedding,612.347288
scoring,2.117961
export,8.594849


,reference_vectors,posts,threads,selected_occurrences
anchor,,,,
muzat,0,0,0,0
muzzies,100,100,100,100
pislam,2,2,2,2
mohammedan,13,13,13,13
mudslime,100,100,100,100
turk,100,100,100,100


muzzies,,100
turk,,100
mudslime,,100
mohammedan,,13
pislam,,2
muzat,,0


## 2. Retrieved corpus

Counts refer to the consumed input prefix, not necessarily the whole source file.
Excluded reference-thread posts are counted without parsing their text. Copied-text exclusions
apply to the remaining examined posts. A seed match indicates retrieval relevance, not a hate label.
The full raw source post is retained in `raw_post` for later work; no conversation is reconstructed.

In [51]:
display(pd.Series(run_info["retrieval"]["counts"]).to_frame("count"))
print("Stopped at:", run_info["retrieval"]["stop_reason"])
print("Selection:", run_info["retrieval"]["selection"])
display(tables["retrieval_summary"])
display(HTML(bar_chart_html(tables["retrieval_summary"].set_index("seed")["selected_posts"],
                            "Selected posts per retrieval seed (counts can overlap)")))
post_lengths = posts["text"].str.len()
display(post_lengths.describe().to_frame("text_characters"))
display(HTML(bar_chart_html(pd.cut(post_lengths, bins=12).value_counts(sort=False),
                            "Post lengths in characters: counts per interval")))
post_dates = pd.to_datetime(pd.to_numeric(posts["timestamp"], errors="coerce"), unit="s", utc=True)
print("UTC coverage:", post_dates.min(), "to", post_dates.max())
print("Posts with unknown timestamp:", int(post_dates.isna().sum()))
with pd.option_context("display.max_colwidth", 240):
    display(posts[["post_key", "thread_key", "timestamp", "matched_seeds", "text"]].head())

,count
source_records_read,7783
posts_examined,301048
reference_threads_excluded,0
reference_thread_posts_excluded,0
reference_text_posts_excluded,1
empty_text_posts,13504
seed_matched_posts,10000
selected_posts,10000
selected_threads,1860
threads_with_reference_text_exclusions,1


Stopped at: maximum_posts
Selection: First matching posts in source order, after reference exclusions; not random sampling


,seed,selected_posts
0,islam,2548
1,muslim,3385
2,muslims,4322
3,islamic,850
4,mosque,144
5,mosques,156
6,quran,203
7,koran,79
8,islamophobia,88


islam,,2548
muslim,,3385
muslims,,4322
islamic,,850
mosque,,144
mosques,,156
quran,,203
koran,,79
islamophobia,,88


,text_characters
count,10000.000000
mean,247.309600
std,293.948765
min,5.000000
25%,81.000000
50%,151.000000
75%,288.000000
max,1997.000000


"(3.008, 171.0]",,5532
"(171.0, 337.0]",,2431
"(337.0, 503.0]",,921
"(503.0, 669.0]",,420
"(669.0, 835.0]",,242
"(835.0, 1001.0]",,112
"(1001.0, 1167.0]",,91
"(1167.0, 1333.0]",,56
"(1333.0, 1499.0]",,49
"(1499.0, 1665.0]",,43
"(1665.0, 1831.0]",,44


UTC coverage: 2016-07-11 18:12:56+00:00 to 2016-07-15 13:00:48+00:00
Posts with unknown timestamp: 0


,post_key,thread_key,timestamp,matched_seeds,text
0,4chan:pol:80664361,4chan:pol:80664361,1468284516,"[muslim, muslims]","This will probably get lost in the shuffle, but here goes.\nshall make no law respecting an establishment of religion\n\n>There might not be a budget, but Illinois could become the first state with a law on the books that gives Muslims ..."
1,4chan:pol:80664677,4chan:pol:80664361,1468284699,[muslim],">relations between Illinois and Muslim-majority countries\n\nhold up, hold up, isn't international relations one of the, like, three things that the federal government is actually allowed to handle and the states are barred from touching?"
2,4chan:pol:80666661,4chan:pol:80664361,1468285848,[muslims],"Dude, Quebec is so full of Muslims now they almost passed Bill 51\nBut because of Trudope the same legislation will probably go national."
3,4chan:pol:80663539,4chan:pol:80660814,1468284044,[islam],* Gun Control is really hard:\nThis country only came into existence because the British tried to confiscate arms. The result was we shot their asses and sent them packing. Strongest military in the world at that time.\n\nSelf defense i...
4,4chan:pol:80663833,4chan:pol:80660814,1468284212,[muslims],"Apart from deporting 11million illegals\nTHOSE ARE PISS EASY GOALS\nA WALL IS FUCKING EASY TO BUILD\nBANNING A FAITH IS EASY TOO\nYOU SAY FUCK OFF NO MUSLIMS.\nRefuse entry to burkas and terrorists, put pork all over the gates."


## 3. Candidate generation

The existing baseline removes stopwords **before** constructing n-grams. A candidate such as
`generations holocaust` can therefore span `generations after the Holocaust` in the source.
This run retains that behavior and flags such occurrences instead of silently replacing the algorithm.
Word spans address the exact saved text. TF-IDF selection remains strictly above the mean of
each term's maximum TF-IDF across posts; the POS rules remain the existing provisional rules.

In [52]:
ngram_counts = pd.DataFrame.from_dict(candidate_stage_counts["ngrams"], orient="index")
ngram_counts.index.name = "ngram_length"
display(ngram_counts)
display(pd.Series({key: value for key, value in candidate_stage_counts.items() if key != "ngrams"}))
tfidf_candidates = tables["tfidf_candidates"]
display(tfidf_candidates.groupby("ngram_length").agg(
    terms=("candidate", "size"), selected=("selected", "sum")
))
display(HTML(bar_chart_html(tfidf_candidates.loc[tfidf_candidates["selected"], "ngram_length"]
                            .value_counts().sort_index(), "TF-IDF-selected candidates by n-gram length")))
print("TF-IDF threshold:", run_info["tfidf_threshold"])
display(tfidf_candidates.sort_values("max_tfidf", ascending=False).head(20))
noncontiguous_occurrences = occurrences.loc[~occurrences["is_contiguous"]]
with pd.option_context("display.max_colwidth", 160):
    display(noncontiguous_occurrences[["candidate", "matched_text", "post_key", "word_spans"]].head(15))

,before_pos_occurrences,after_pos_occurrences,before_pos_unique,after_pos_unique,removed_examples
ngram_length,,,,,
1,293793,146751,22084,17116,"[!, #, $, %, &, ', '', '99, 'abdul, 'ali]"
2,283804,60561,161468,41723,"[! !, ! #, ! ', ! '', ! 'd, ! 'll, ! 'm, ! 're..."
3,273828,28479,235680,24693,"[! ! !, ! ! '', ! ! 's, ! ! 've, ! ! ,, ! ! ....."


ngram_policy                 legacy_stopwords_before_ngrams
tfidf_vocabulary                                      83532
tfidf_selected                                        33976
tfidf_threshold                                     0.22667
matched_candidates                                    33976
unmatched_candidates                                     []
candidate_occurrences                                172742
noncontiguous_occurrences                             18509
candidate_post_embeddings                            154634
ranked_candidates                                     33976
relevance_survivors                                   17611
dtype: object

,terms,selected
ngram_length,,
1,17116,9018
2,41723,16305
3,24693,8653


1,,9018
2,,16305
3,,8653


TF-IDF threshold: 0.22666990898683906


,candidate,max_tfidf,selected,ngram_length
38612,kek,1.0,True,1
72104,sure,1.0,True,1
41486,link,1.0,True,1
48861,muslims,1.0,True,1
35866,islam,1.0,True,1
22390,ex-muslim,1.0,True,1
47543,muslim,1.0,True,1
40934,level,1.0,True,1
22392,ex-muslims,1.0,True,1
36690,islamic,1.0,True,1


,candidate,matched_text,post_key,word_spans
44,relations illinois,relations between Illinois,4chan:pol:80664361,"[[499, 508], [517, 525]]"
45,illinois muslim-majority,Illinois and Muslim-majority,4chan:pol:80664361,"[[517, 525], [530, 545]]"
47,relations illinois,relations between Illinois,4chan:pol:80664361,"[[1078, 1087], [1096, 1104]]"
48,illinois muslim-majority,Illinois and Muslim-majority,4chan:pol:80664361,"[[1096, 1104], [1109, 1124]]"
50,relations illinois muslim-majority,relations between Illinois and Muslim-majority,4chan:pol:80664361,"[[499, 508], [517, 525], [530, 545]]"
51,illinois muslim-majority countries,Illinois and Muslim-majority countries,4chan:pol:80664361,"[[517, 525], [530, 545], [546, 555]]"
52,relations illinois muslim-majority,relations between Illinois and Muslim-majority,4chan:pol:80664361,"[[1078, 1087], [1096, 1104], [1109, 1124]]"
53,illinois muslim-majority countries,Illinois and Muslim-majority countries,4chan:pol:80664361,"[[1096, 1104], [1109, 1124], [1125, 1134]]"
63,relations illinois,relations between Illinois,4chan:pol:80664677,"[[1, 10], [19, 27]]"
64,illinois muslim-majority,Illinois and Muslim-majority,4chan:pol:80664677,"[[19, 27], [32, 47]]"


## 4. Semantic ranking and inspection views

Each candidate/post score averages five nearest reference vectors; several can represent the
same anchor. Candidate relevance averages those scores across source posts. The relevance cutoff
is the mean candidate score. Singletons remain in the main ranking.

Seed/anchor exclusion matches whole phrases with word boundaries, ignoring case; it does not
remove unrelated substrings. `original_rank` always refers to the full ranking.
`display_position` describes a filtered view only. These views do not rescore candidates.

In [53]:
print("HMApp relevance cutoff:", run_info["relevance_threshold"])
print("Above cutoff:", int(rankings["passes_relevance"].sum()), "/", len(rankings))
display(HTML(bar_chart_html(pd.cut(rankings["relevance"], bins=20).value_counts(sort=False),
                            f'Relevance distribution; cutoff = {run_info["relevance_threshold"]:.6f}')))
display(HTML(bar_chart_html(pd.cut(rankings["n_posts"], bins=[0, 1, 2, 5, 10, float("inf")])
                            .value_counts(sort=False), "Distinct scored posts per candidate")))
display(rankings.head(30))

HMApp relevance cutoff: 0.5272097587585449
Above cutoff: 17611 / 33976


"(0.273, 0.301]",,12
"(0.301, 0.329]",,36
"(0.329, 0.357]",,155
"(0.357, 0.385]",,585
"(0.385, 0.413]",,1363
"(0.413, 0.441]",,2283
"(0.441, 0.469]",,3151
"(0.469, 0.497]",,3900
"(0.497, 0.525]",,4526
"(0.525, 0.553]",,4875
"(0.553, 0.581]",,4886


"(0.0, 1.0]",,24230
"(1.0, 2.0]",,3368
"(2.0, 5.0]",,2819
"(5.0, 10.0]",,1455
"(10.0, inf]",,2104


,original_rank,candidate,relevance,n_posts,n_occurrences,n_threads,n_contiguous_occurrences,n_noncontiguous_occurrences,passes_relevance
0,1,mudslime immigrants,0.833070,1,1,1,1,0,True
1,2,open mudslime,0.827587,1,1,1,0,1,True
2,3,open mudslime immigrants,0.816781,1,1,1,0,1,True
3,4,mudslime rampage,0.813084,2,2,1,2,0,True
4,5,mudslime mudslime,0.812068,1,1,1,0,1,True
5,6,mudslime,0.811279,35,37,32,37,0,True
6,7,mudslime shit,0.809392,1,1,1,1,0,True
7,8,bean,0.808781,1,1,1,1,0,True
8,9,muzzies,0.804976,48,48,42,48,0,True
9,10,death mudslimes,0.800928,1,1,1,0,1,True


In [54]:
seed_excluded_view = inspection_view(rankings, excluded_phrases=seed_words)
anchor_excluded_view = inspection_view(rankings, excluded_phrases=anchor_variants)
two_post_view = inspection_view(rankings, minimum_posts=2)
two_thread_view = inspection_view(rankings, minimum_threads=2)

for view_name, view in [
    ("Without seeds or seed-containing phrases", seed_excluded_view),
    ("Without anchors or anchor-containing phrases", anchor_excluded_view),
    ("At least two scored posts", two_post_view),
    ("At least two source threads", two_thread_view),
]:
    print(view_name, f"({len(view):,} candidates)")
    display(view.head(20))

Without seeds or seed-containing phrases (27,450 candidates)


,display_position,original_rank,candidate,relevance,n_posts,n_occurrences,n_threads,n_contiguous_occurrences,n_noncontiguous_occurrences,passes_relevance
0,1,1,mudslime immigrants,0.833070,1,1,1,1,0,True
1,2,2,open mudslime,0.827587,1,1,1,0,1,True
2,3,3,open mudslime immigrants,0.816781,1,1,1,0,1,True
3,4,4,mudslime rampage,0.813084,2,2,1,2,0,True
4,5,5,mudslime mudslime,0.812068,1,1,1,0,1,True
5,6,6,mudslime,0.811279,35,37,32,37,0,True
6,7,7,mudslime shit,0.809392,1,1,1,1,0,True
7,8,8,bean,0.808781,1,1,1,1,0,True
8,9,9,muzzies,0.804976,48,48,42,48,0,True
9,10,10,death mudslimes,0.800928,1,1,1,0,1,True


Without anchors or anchor-containing phrases (33,923 candidates)


,display_position,original_rank,candidate,relevance,n_posts,n_occurrences,n_threads,n_contiguous_occurrences,n_noncontiguous_occurrences,passes_relevance
0,1,8,bean,0.808781,1,1,1,1,0,True
1,2,10,death mudslimes,0.800928,1,1,1,0,1,True
2,3,25,time mudslimes,0.781418,1,1,1,0,1,True
3,4,28,mudslimes,0.778568,42,46,42,46,0,True
4,5,29,thing mudslimes,0.777383,1,1,1,0,1,True
5,6,32,stupider mudslimes,0.775577,1,1,1,0,1,True
6,7,35,muzzie muzzie,0.774046,1,1,1,0,1,True
7,8,37,language mudslimes,0.773760,1,1,1,0,1,True
8,9,42,whites time mudslimes,0.765997,1,1,1,0,1,True
9,10,43,muzzie countries,0.765930,1,1,1,1,0,True


At least two scored posts (9,746 candidates)


,display_position,original_rank,candidate,relevance,n_posts,n_occurrences,n_threads,n_contiguous_occurrences,n_noncontiguous_occurrences,passes_relevance
0,1,4,mudslime rampage,0.813084,2,2,1,2,0,True
1,2,6,mudslime,0.811279,35,37,32,37,0,True
2,3,9,muzzies,0.804976,48,48,42,48,0,True
3,4,20,fucking mudslime rampage,0.786898,2,2,1,2,0,True
4,5,22,turk,0.784746,13,17,12,17,0,True
5,6,26,fuck mudslime,0.780746,2,2,2,0,2,True
6,7,28,mudslimes,0.778568,42,46,42,46,0,True
7,8,54,mudslim,0.755967,3,3,3,3,0,True
8,9,65,muzzie,0.747034,16,18,16,18,0,True
9,10,79,dead muzzie,0.720584,2,2,2,2,0,True


At least two source threads (8,923 candidates)


,display_position,original_rank,candidate,relevance,n_posts,n_occurrences,n_threads,n_contiguous_occurrences,n_noncontiguous_occurrences,passes_relevance
0,1,6,mudslime,0.811279,35,37,32,37,0,True
1,2,9,muzzies,0.804976,48,48,42,48,0,True
2,3,22,turk,0.784746,13,17,12,17,0,True
3,4,26,fuck mudslime,0.780746,2,2,2,0,2,True
4,5,28,mudslimes,0.778568,42,46,42,46,0,True
5,6,54,mudslim,0.755967,3,3,3,3,0,True
6,7,65,muzzie,0.747034,16,18,16,18,0,True
7,8,79,dead muzzie,0.720584,2,2,2,2,0,True
8,9,81,mohammedanism,0.719177,2,2,2,2,0,True
9,10,97,libshits,0.711205,3,3,3,3,0,True


## 5. Inspect an individual candidate

Change `selected_term` to a candidate from the tables. The highlights use the saved constituent
word offsets, so a noncontiguous candidate highlights its words rather than pretending the
intervening text is part of the extracted expression. The preview shows one occurrence per
supporting post, up to five posts; the occurrence table preserves all occurrences.

In [55]:
selected_term = rankings.iloc[0]["candidate"]  # Replace with a candidate string to inspect.
selected_ranking = rankings.loc[rankings["candidate"].eq(selected_term)]
if selected_ranking.empty:
    raise ValueError(f"Candidate not found in this run: {selected_term!r}")
display(selected_ranking)
selected_occurrences = occurrences.loc[occurrences["candidate"].eq(selected_term)]
display(selected_occurrences[["post_key", "thread_key", "matched_text", "start", "end", "is_contiguous"]])
source_posts = posts.set_index("post_key")
selected_post_scores = post_scores.loc[post_scores["candidate"].eq(selected_term)].set_index("post_key")
for occurrence in selected_occurrences.drop_duplicates("post_key").head(5).itertuples(index=False):
    source_post = source_posts.loc[occurrence.post_key]
    print("Post:", occurrence.post_key, "Thread:", occurrence.thread_key,
          "Relevance:", selected_post_scores.loc[occurrence.post_key, "relevance"])
    display(HTML(highlight_word_spans(source_post["text"], occurrence.word_spans)))

,original_rank,candidate,relevance,n_posts,n_occurrences,n_threads,n_contiguous_occurrences,n_noncontiguous_occurrences,passes_relevance
0,1,mudslime immigrants,0.83307,1,1,1,1,0,True


,post_key,thread_key,matched_text,start,end,is_contiguous
81074,4chan:pol:81028155,4chan:pol:81025378,mudslime immigrants,120,139,True


Post: 4chan:pol:81028155 Thread: 4chan:pol:81025378 Relevance: 0.8330701


## 6. Summary diagnostics for the report

These are descriptive observations, not precision/recall/F1 or proof of hateful meaning.
Distinct posts and threads are proxies for support, not proof of independent adoption.
Reference coverage concerns the frozen bank; missing anchors are not filled from discovery.

In [56]:
top_candidates = rankings.head(50)
seed_pattern = phrase_pattern(seed_words)
anchor_pattern = phrase_pattern(anchor_variants)
diagnostics = pd.Series({
    "ranked_candidates": len(rankings),
    "singleton_candidates": int(rankings["n_posts"].eq(1).sum()),
    "singleton_fraction": float(rankings["n_posts"].eq(1).mean()),
    "top_50_actual_size": len(top_candidates),
    "top_50_containing_seeds": int(top_candidates["candidate"].map(lambda text: bool(seed_pattern.search(text))).sum()),
    "top_50_containing_anchors": int(top_candidates["candidate"].map(lambda text: bool(anchor_pattern.search(text))).sum()),
    "anchors_represented": int(reference_coverage["reference_vectors"].gt(0).sum()),
    "anchors_in_vocabulary": len(reference_coverage),
    "noncontiguous_occurrences": len(noncontiguous_occurrences),
    "candidates_with_noncontiguous_occurrences": int(rankings["n_noncontiguous_occurrences"].gt(0).sum()),
})
display(diagnostics.to_frame("value"))
for limitation in run_info["limitations"]:
    print("-", limitation)

,value
ranked_candidates,33976.00000
singleton_candidates,24230.00000
singleton_fraction,0.71315
top_50_actual_size,50.00000
top_50_containing_seeds,2.00000
top_50_containing_anchors,35.00000
anchors_represented,5.00000
anchors_in_vocabulary,6.00000
noncontiguous_occurrences,18509.00000
candidates_with_noncontiguous_occurrences,13545.00000


- Consecutive historical source slice; retrieval is seed-conditioned and stops in file order
- Reference anchor matches are assumed suitable contexts, not validated hate labels
- Thread holdout is defined by this bank's selected source threads, not its entire sampling pool
- Normalized full-text equality excludes copies, not shared passages or paraphrases
- Legacy stopword-before-ngram extraction retained; noncontiguous occurrences are flagged
- Existing noun/adjective POS rules are provisional, not verified exact Wu rules
- Reference preparation selects one occurrence per anchor/thread and caps contexts; candidate repeats are averaged
- Distinct posts/threads do not establish independent adoption; similarity is not a hate label


### Notes to write after inspecting the actual outputs

- Which candidates appear meaningful in their original posts? Which look malformed?
- How much does the visible list change after hiding known seeds/anchors, while scores stay fixed?
- How much support comes from multiple posts and multiple threads?
- Which missing or dominant anchors could limit interpretation?

Record examples and uncertainties. A high cosine score alone does not establish hateful usage
or that an expression is historically new. Evaluation and conversational-context methods are separate tasks.